In [ ]:
import pandas as pd
from jax import numpy as jnp
from jax.random import PRNGKey

pd.options.plotting.backend = "plotly"

from summer3.graph import CompartmentValues, defer, Parameter
from summer3.epi import TransitionFlow, Stratification, CompartmentMap, CompartmentalEpiModel, CategoryData, CompartmentalModelODE, build_istate, dti_to_epoch

In [ ]:
start_time = 0
end_time = 70
model_comps = [
    "vaccinated", 
    "susceptible", 
    "infectious_asympt", 
    "infectious_sympt", 
    "recovered",
]
infect_comps = [
    "infectious_asympt", 
    "infectious_sympt",
]
disease_state = Stratification("disease_state", model_comps)
humans = CompartmentMap.new(disease_state)

In [ ]:
def infection_process(compartment_values, disease_state, contact_rate):
    infectious_pop = compartment_values.query(disease_state["infectious_asympt", "infectious_sympt"]).sum()
    total_pop = compartment_values.sum()
    return infectious_pop / total_pop * contact_rate

force_of_infection = defer(infection_process)(
    CompartmentValues, 
    disease_state, 
    Parameter("contact_rate", 0.0),
)
infection = TransitionFlow(
    "infection", 
    disease_state["susceptible"], 
    disease_state["infectious_asympt"], 
    force_of_infection,
)
progression = TransitionFlow(
    "progression",
    disease_state["infectious_asympt"],
    disease_state["infectious_sympt"],
    1.0 / Parameter("asympt_time", 0.0),
)
recovery = TransitionFlow(
    "recovery",
    disease_state["infectious_asympt"],
    disease_state["recovered"],
    1.0 / Parameter("recovery_time", 0.0),
)

times = pd.date_range("7 jun 1980", "7 december 1980")
epi_model = CompartmentalEpiModel(humans, times)
init_pops = CategoryData(disease_state.categories(), jnp.array(([0.0, 0.99, 0.01, 0.0, 0.0])))
epi_model.set_initial_population(init_pops)
epi_model.add_flow(infection)
epi_model.add_flow(progression)
epi_model.add_flow(recovery)

In [ ]:
params = {"contact_rate": 0.5, "asympt_time": 4.0, "recovery_time": 4.0}
results = epi_model.run(params)

In [ ]:
results["compartments"].to_pandas_df().plot()